In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset, ClassificationPreset

print("Librerías cargadas")

Librerías cargadas


In [2]:
PROJECT_ROOT   = Path(".").resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"
REPORTS_DIR    = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_parquet(DATA_PROCESSED / "X_train.parquet")
X_test  = pd.read_parquet(DATA_PROCESSED / "X_test.parquet")

y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / "y_test.parquet").squeeze()

model = joblib.load(PROJECT_ROOT / "models" / "LightGBM_best.joblib")

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print("Modelo cargado: LightGBM")

X_train: (413378, 422)
X_test:  (88581, 422)
Modelo cargado: LightGBM


In [3]:
# Recuperar nombres de columnas del preprocesador
preprocessor  = joblib.load(PROJECT_ROOT / "models" / "preprocessor.joblib")
num_cols      = preprocessor.transformers_[0][2]
cat_low       = preprocessor.transformers_[1][2]
cat_high      = preprocessor.transformers_[2][2]
feature_names = num_cols + cat_low + cat_high

# Asignar nombres de columnas
X_train.columns = feature_names
X_test.columns  = feature_names

# Agregar target y predicciones
train_df = X_train.copy()
test_df  = X_test.copy()

train_df["target"]     = y_train.values
train_df["prediction"] = model.predict_proba(X_train)[:, 1]

test_df["target"]      = y_test.values
test_df["prediction"]  = model.predict_proba(X_test)[:, 1]

print(f"Train con target y predicción: {train_df.shape}")
print(f"Test con target y predicción:  {test_df.shape}")

Train con target y predicción: (413378, 424)
Test con target y predicción:  (88581, 424)


In [4]:
from evidently import BinaryClassification

train_sample = train_df.sample(n=10000, random_state=42)

data_definition = DataDefinition(
    classification=[
        BinaryClassification(
            target="target",
            prediction_probas="prediction"
        )
    ]
)

ref_dataset  = Dataset.from_pandas(train_sample, data_definition=data_definition)
curr_dataset = Dataset.from_pandas(test_df,      data_definition=data_definition)

drift_report = Report(metrics=[DataDriftPreset()])

# run devuelve un Snapshot — ese objeto tiene save_html
snapshot = drift_report.run(current_data=curr_dataset, reference_data=ref_dataset)
snapshot.save_html(str(REPORTS_DIR / "drift_report.html"))

print("Reporte guardado en reports/drift_report.html")

Reporte guardado en reports/drift_report.html


In [5]:
classification_report = Report(metrics=[ClassificationPreset()])

snapshot_clf = classification_report.run(
    current_data=curr_dataset,
    reference_data=ref_dataset
)

snapshot_clf.save_html(str(REPORTS_DIR / "classification_report.html"))
print("Reporte guardado en reports/classification_report.html")

Reporte guardado en reports/classification_report.html


In [7]:
# Extraer resultados del snapshot
results = snapshot_clf.dict()

# Extraer métricas por nombre
metricas = {m["metric_name"].split("(")[0]: m["value"] for m in results["metrics"]}

roc_auc = metricas["RocAuc"]
f1      = metricas["F1Score"]

# Umbrales de alerta
UMBRAL_ROC_AUC = 0.90
UMBRAL_F1      = 0.60

print(f"ROC AUC actual: {roc_auc:.4f}  | Umbral: {UMBRAL_ROC_AUC}")
print(f"F1 actual:      {f1:.4f}  | Umbral: {UMBRAL_F1}")

if roc_auc < UMBRAL_ROC_AUC or f1 < UMBRAL_F1:
    print("\nALERTA: El modelo degradó — se recomienda reentrenar")
else:
    print("\nEstado: El modelo está dentro de parámetros aceptables")

ROC AUC actual: 0.9494  | Umbral: 0.9
F1 actual:      0.6944  | Umbral: 0.6

Estado: El modelo está dentro de parámetros aceptables


In [8]:
import json
from datetime import datetime

estado_monitoring = {
    "fecha":           datetime.now().strftime("%Y-%m-%d %H:%M"),
    "roc_auc":         round(roc_auc, 4),
    "f1":              round(f1, 4),
    "umbral_roc_auc":  UMBRAL_ROC_AUC,
    "umbral_f1":       UMBRAL_F1,
    "drift_detectado": False,
    "reentrenar":      roc_auc < UMBRAL_ROC_AUC or f1 < UMBRAL_F1
}

MODELS_DIR_REAL = PROJECT_ROOT / "models"

with open(MODELS_DIR_REAL / "monitoring_status.json", "w") as f:
    json.dump(estado_monitoring, f, indent=2)

print(f"Guardado en: {MODELS_DIR_REAL / 'monitoring_status.json'}")
print(json.dumps(estado_monitoring, indent=2))

Guardado en: C:\Users\micke\OneDrive\Desktop\projects\Frauddetection-mlops\models\monitoring_status.json
{
  "fecha": "2026-03-29 10:07",
  "roc_auc": 0.9494,
  "f1": 0.6944,
  "umbral_roc_auc": 0.9,
  "umbral_f1": 0.6,
  "drift_detectado": false,
  "reentrenar": false
}
